# short_volshock_sell_guard_official_replay_v1

## Goal

The previous official replay proved the lagged `short_volshock` cap is a near-miss, not a promotion candidate: the signal frequently reduced the target, but `min_rebalance` and `trade_lock` absorbed most reductions before they became orders. This notebook tests one explicit mechanism: a default-off sell-side execution guard that only relaxes execution gates when the overlay cap creates a real de-risking sell step.

## Governance

- Baseline: `v19reg_v22_budget_usage_bull_guard_v1`
- Direct comparison: `short_horizon_risk_cap_counterfactual_v1`, `short_volshock_official_replay_screen_v1`
- Role: `diagnose`
- Changed layer: `execution` / `model_selection` / `risk_state_representation`
- Target windows: `wf_2022`, `wf_2023_2024`
- Acceptance: official warm-start daily path must clear `relative_return >= +30pp` and `MaxDD <= 15%` on both folds; decision traces must show de-risk orders are not mostly absorbed by `min_rebalance` / `trade_lock`
- Promotion: blocked even if this passes; this card uses regenerated diagnostic BC actors and still needs fixed-split / `wf_2020` / actor-retention review
- Failure mode: the guard increases order count but gives back return, causes over-defensive behavior, or still misses the `+30pp / <=15% MaxDD` joint gate

## External basis

- Qlib official backtest and dataset handler pattern: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL risk-aware trading evaluation reference: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Behavior cloning / covariate shift caution: https://proceedings.mlr.press/v15/ross11a.html
- Risk-sensitive sequential decision framing: https://arxiv.org/abs/1312.4077


In [1]:

from __future__ import annotations

import json
import math
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics, max_drawdown_interval_metrics
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.core.metric_reconciliation import build_warm_start_replay_segment, slice_warm_start_daily_path
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, flatten_feature_frame, init_qlib
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = "short_volshock_sell_guard_official_replay_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
DAILY_DIR = CARD / "daily_paths"
TRACE_DIR = CARD / "decision_traces"
for directory in (DAILY_DIR, TRACE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
    for stale in directory.glob("*.csv"):
        stale.unlink()

SOURCE_CARD = ROOT / "runtime/model_selection/card/state_conditioned_multi_objective_bc_screen_v1"
RISK_CARD = ROOT / "runtime/model_selection/card/drawdown_interval_risk_state_coverage_v1"
COUNTERFACTUAL_CARD = ROOT / "runtime/model_selection/card/short_horizon_risk_cap_counterfactual_v1"
FRAME_PATH = ROOT / "data" / "soxl_profile_balanced_rule_residual_v22.pkl"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
SELECTED_CANDIDATE = "multi_obj_floor45_cap20_riskfirst"
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)
SAMPLE_WEIGHT_CONFIG = StateObjectiveBCSampleWeightConfig(
    rebound_multiplier=2.5,
    downtrend_multiplier=3.5,
    ordinary_multiplier=1.0,
    objective_weight_power=0.15,
    class_balanced_beta=0.99,
    weight_cap=25.0,
)
WALK_FORWARD_WINDOWS = {
    "wf_2022": {"target_fold": "wf_2022_train", "train": ("2010-03-11", "2020-12-31"), "valid": ("2021-01-01", "2021-12-31"), "test": ("2022-01-01", "2022-12-31")},
    "wf_2023_2024": {"target_fold": "wf_2023_2024_train", "train": ("2010-03-11", "2022-12-31"), "valid": ("2023-01-01", "2023-12-31"), "test": ("2024-01-01", "2024-12-31")},
}
OVERLAY_VARIANTS = [
    {"variant": "baseline_no_overlay", "overlay_config": None},
    {"variant": "short_volshock_cap05_hit2_no_guard", "target_cap": 0.05, "min_hits": 2},
    {
        "variant": "short_volshock_cap05_hit2_sell_guard_zero",
        "target_cap": 0.05,
        "min_hits": 2,
        "de_risk_guard_enabled": True,
        "de_risk_rebalance_threshold_override": 0.0,
        "de_risk_trade_lock_override": 0,
    },
    {
        "variant": "short_volshock_cap05_hit2_sell_guard_1pct",
        "target_cap": 0.05,
        "min_hits": 2,
        "de_risk_guard_enabled": True,
        "de_risk_rebalance_threshold_override": 0.01,
        "de_risk_trade_lock_override": 0,
    },
]
EXTERNAL_SOURCES = [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://proceedings.mlr.press/v15/ross11a.html",
    "https://arxiv.org/abs/1312.4077",
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.floating):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def gate_loss(relative_return: float, max_dd: float) -> float:
    return max(0.0, TARGET_RELATIVE_RETURN - float(relative_return)) + 10.0 * max(0.0, float(max_dd) - TARGET_MAX_DD)


def fold_subset(targets: pd.DataFrame, spec: dict[str, Any]) -> pd.DataFrame:
    dates = pd.to_datetime(targets["datetime"]).dt.normalize()
    return targets.loc[
        targets["fold"].astype(str).eq(str(spec["target_fold"]))
        & (dates >= pd.Timestamp(spec["train"][0]).normalize())
        & (dates <= pd.Timestamp(spec["train"][1]).normalize())
    ].sort_values("datetime").reset_index(drop=True)

source_summary = load_json(SOURCE_CARD / "summary.json")
counterfactual_summary = load_json(COUNTERFACTUAL_CARD / "summary.json")
profile = load_profile(PROFILE_NAME)
train_frame = pd.read_pickle(FRAME_PATH)
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
selected_targets = pd.read_csv(SOURCE_CARD / f"{SELECTED_CANDIDATE}_target_contract.csv", parse_dates=["datetime"])
thresholds = pd.read_csv(RISK_CARD / "fold_local_signal_thresholds.csv").set_index("fold")
init_qlib(QLIB_DATA_PATH)
print("source", source_summary["decision"], source_summary["selected_candidate"])
print("counterfactual", counterfactual_summary["decision"], counterfactual_summary["selected_lagged_variant"])
print("targets", selected_targets.shape, selected_targets["fold"].value_counts().to_dict())


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[93639:MainThread](2026-05-26 19:36:21,226) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[93639:MainThread](2026-05-26 19:36:21,920) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[93639:MainThread](2026-05-26 19:36:21,921) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


source state_conditioned_multi_objective_drawdown_failed_no_promotion multi_obj_floor45_cap20_riskfirst
counterfactual diagnostic_full_gate_counterfactual_needs_official_replay_no_promotion {'candidate': 'multi_obj_floor45_cap20_riskfirst', 'cap': 0.05, 'drawdown_margin_pp': 3.0692124897401776, 'fold_count': 2, 'full_gate_all': True, 'full_gate_count': 2, 'gate_loss': 0.0, 'max_max_dd': 11.930787510259822, 'mean_avg_pos': 0.12147280074235572, 'mean_interval_signal_share': 0.7923465423465423, 'mean_relative_return': 56.25703718518331, 'min_relative_return': 31.884600144954902, 'nonzero_cap': True, 'relative_margin_pp': 1.8846001449549021, 'robust_margin_loss': 0.0, 'signal_family': 'short_volshock', 'simple_signal': True, 'timing': 'lag1', 'total_capped_days': 337, 'total_changed_days': 337, 'total_signal_days': 367, 'worst_interval_avg_pos': 0.15010328601292455}
targets (1009, 46) {'wf_2023_2024_train': 756, 'wf_2022_train': 253}


In [2]:

def overlay_for_fold(fold: str, *, target_cap: float, min_hits: int, **guard_config: Any) -> dict[str, Any]:
    row = thresholds.loc[fold]
    config = {
        "type": "short_volshock_action_cap",
        "realized_vol_21_min": float(row["realized_vol_21_q80"]),
        "vol_of_vol_21_min": float(row["vol_of_vol_21_q80"]),
        "downside_semivol_21_min": float(row["downside_semivol_21_q80"]),
        "short_volshock_min_feature_hits": int(min_hits),
        "target_cap": float(target_cap),
    }
    config.update({key: value for key, value in guard_config.items() if value is not None})
    return config

fold_cache: dict[str, dict[str, Any]] = {}
segment_rows: list[dict[str, Any]] = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    feature_full = flatten_feature_frame(dataset.handler.feature_frame).droplevel("instrument").copy()
    price_full = dataset.handler.price_frame.droplevel("instrument")["price"].sort_index()
    warm_seg = build_warm_start_replay_segment(feature_full.index, eval_seg=spec["test"], lookback=LOOKBACK)
    fold_cache[fold_name] = {"dataset": dataset, "feature_full": feature_full, "price_full": price_full, "warm_seg": warm_seg}
    segment_rows.append({
        "fold": fold_name,
        "feature_start": str(feature_full.index.min().date()),
        "feature_end": str(feature_full.index.max().date()),
        "test_start": spec["test"][0],
        "test_end": spec["test"][1],
        "warm_start": warm_seg[0],
        "warm_end": warm_seg[1],
    })
segments = pd.DataFrame(segment_rows)
segments.to_csv(CARD / "warm_start_segments.csv", index=False)
display(segments)


[93639:MainThread](2026-05-26 19:36:24,500) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[93639:MainThread](2026-05-26 19:36:24,503) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[93639:MainThread](2026-05-26 19:36:24,503) INFO - qlib.timer - [log.py:127] - Time cost: 0.005s | Init data Done


[93639:MainThread](2026-05-26 19:36:27,188) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[93639:MainThread](2026-05-26 19:36:27,189) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[93639:MainThread](2026-05-26 19:36:27,191) INFO - qlib.timer - [log.py:127] - Time cost: 0.004s | Init data Done


,fold,feature_start,feature_end,test_start,test_end,warm_start,warm_end
0,wf_2022,2011-03-11,2022-12-30,2022-01-01,2022-12-31,2021-10-07,2022-12-30
1,wf_2023_2024,2011-03-11,2024-12-31,2024-01-01,2024-12-31,2023-10-05,2024-12-31


In [3]:

metric_rows: list[dict[str, Any]] = []
trace_summary_rows: list[dict[str, Any]] = []
daily_manifest_rows: list[dict[str, Any]] = []
fit_rows: list[dict[str, Any]] = []

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    fold_targets = fold_subset(selected_targets, spec)
    if fold_targets.empty:
        raise ValueError(f"empty targets for {fold_name}")
    set_seed(SEED + len(fold_name))
    fit = fit_state_objective_bc_actor_from_frame(
        train_frame,
        targets=fold_targets,
        profile=profile,
        train_config=BC_TRAIN_CONFIG,
        sample_weight_config=SAMPLE_WEIGHT_CONFIG,
        sample_size=0,
        rng_seed=SEED,
    )
    fit_rows.append({
        "candidate": SELECTED_CANDIDATE,
        "fold": fold_name,
        "target_rows": int(len(fold_targets)),
        "bc_samples": int(len(fit.actions)),
        "before_accuracy": float(fit.metrics.get("before_accuracy", np.nan)),
        "after_accuracy": float(fit.metrics.get("after_accuracy", np.nan)),
        "after_loss": float(fit.metrics.get("after_loss", np.nan)),
        "weight_mean": float(fit.metrics.get("weight_mean", np.nan)) if "weight_mean" in fit.metrics else np.nan,
        "weight_max": float(fit.metrics.get("weight_max", np.nan)) if "weight_max" in fit.metrics else np.nan,
    })
    cache = fold_cache[fold_name]
    for variant in OVERLAY_VARIANTS:
        variant_name = str(variant["variant"])
        overlay_config = None
        if variant_name != "baseline_no_overlay":
            overlay_config = overlay_for_fold(
                fold_name,
                target_cap=float(variant["target_cap"]),
                min_hits=int(variant["min_hits"]),
                de_risk_guard_enabled=variant.get("de_risk_guard_enabled"),
                de_risk_rebalance_threshold_override=variant.get("de_risk_rebalance_threshold_override"),
                de_risk_trade_lock_override=variant.get("de_risk_trade_lock_override"),
                de_risk_trade_lock_cap=variant.get("de_risk_trade_lock_cap"),
            )
        bt = backtest_profile(
            fit.actor,
            cache["feature_full"],
            cache["price_full"],
            position_cap_config=profile.position_cap,
            position_budget_config=profile.position_budget,
            execution_config=profile.execution,
            asset_idx=0,
            feature_columns=feature_columns,
            model_config=profile.model,
            mode_router_config=profile.mode_router,
            test_seg=cache["warm_seg"],
            overlay_config=overlay_config,
            trace_decisions=True,
        )
        warm_daily = slice_warm_start_daily_path(
            portfolio_df=bt["portfolio"],
            price_series=cache["price_full"],
            eval_seg=spec["test"],
            position_history=bt.get("position_history", []),
            trade_signals=bt.get("trade_signals", []),
        )
        daily_path = DAILY_DIR / f"{SELECTED_CANDIDATE}_{fold_name}_{variant_name}_official_daily_path.csv"
        warm_daily.to_csv(daily_path, index=False)
        daily_manifest_rows.append({"candidate": SELECTED_CANDIDATE, "fold": fold_name, "variant": variant_name, "path": str(daily_path.relative_to(ROOT))})
        metrics = daily_path_metrics(warm_daily)
        interval = max_drawdown_interval_metrics(warm_daily)
        trace = pd.DataFrame(bt.get("decision_trace", []))
        trace_path = TRACE_DIR / f"{SELECTED_CANDIDATE}_{fold_name}_{variant_name}_decision_trace.csv"
        trace.to_csv(trace_path, index=False)
        if trace.empty:
            cap_changed = pd.Series(dtype=bool)
            cap_order = pd.Series(dtype=bool)
        else:
            cap_changed = pd.to_numeric(trace.get("target_after_overlay"), errors="coerce") < pd.to_numeric(trace.get("target_before_overlay"), errors="coerce") - 1e-9
            cap_order = cap_changed & (pd.to_numeric(trace.get("order_count"), errors="coerce").fillna(0) > 0)
        trace_summary_rows.append({
            "candidate": SELECTED_CANDIDATE,
            "fold": fold_name,
            "variant": variant_name,
            "trace_path": str(trace_path.relative_to(ROOT)),
            "trace_rows": int(len(trace)),
            "overlay_gate_ready_days": int(trace.get("overlay_gate_ready", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()) if not trace.empty else 0,
            "overlay_sell_side_de_risk_ready_days": int(trace.get("overlay_sell_side_de_risk_ready", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()) if not trace.empty else 0,
            "de_risk_rebalance_applied_days": int(trace.get("de_risk_rebalance_applied", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()) if not trace.empty else 0,
            "de_risk_trade_lock_applied_days": int(trace.get("de_risk_trade_lock_applied", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()) if not trace.empty else 0,
            "cap_changed_days": int(cap_changed.sum()) if not trace.empty else 0,
            "cap_order_days": int(cap_order.sum()) if not trace.empty else 0,
            "blocked_by_trade_lock_days": int(trace.get("blocked_by_trade_lock", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()) if not trace.empty else 0,
            "blocked_by_min_rebalance_days": int(trace.get("blocked_by_min_rebalance", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()) if not trace.empty else 0,
            "blocked_by_lot_rounding_days": int(trace.get("blocked_by_lot_rounding", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()) if not trace.empty else 0,
            "sell_order_days": int((pd.to_numeric(trace.get("delta_shares"), errors="coerce").fillna(0) < 0).sum()) if not trace.empty and "delta_shares" in trace else 0,
        })
        source_match = pd.read_csv(SOURCE_CARD / "warm_start_official_metrics.csv")
        source_row = source_match[source_match["candidate"].eq(SELECTED_CANDIDATE) & source_match["fold"].eq(fold_name)].iloc[0]
        metric_rows.append({
            "candidate": SELECTED_CANDIDATE,
            "fold": fold_name,
            "variant": variant_name,
            "status": "evaluated",
            "requested_start": spec["test"][0],
            "requested_end": spec["test"][1],
            "warm_start": cache["warm_seg"][0],
            "warm_end": cache["warm_seg"][1],
            "official_warm_return": float(metrics["return"]),
            "official_warm_buyhold_return": float(metrics["buy_and_hold_return"]),
            "official_warm_relative_return": float(metrics["relative_return"]),
            "official_warm_max_dd": float(metrics["max_dd"]),
            "official_warm_avg_pos": float(metrics["avg_pos"]),
            "trades": int(bt.get("trades", 0)),
            "full_gate": bool(float(metrics["relative_return"]) >= TARGET_RELATIVE_RETURN and float(metrics["max_dd"]) <= TARGET_MAX_DD),
            "gate_loss": gate_loss(float(metrics["relative_return"]), float(metrics["max_dd"])),
            "peak_date": interval["peak_date"],
            "trough_date": interval["trough_date"],
            "interval_days": interval["interval_days"],
            "interval_avg_pos": interval["interval_avg_pos"],
            "interval_max_pos": interval["interval_max_pos"],
            "source_official_relative_return": float(source_row["official_warm_relative_return"]),
            "source_official_max_dd": float(source_row["official_warm_max_dd"]),
            "relative_delta_vs_source_official": float(metrics["relative_return"] - source_row["official_warm_relative_return"]),
            "max_dd_delta_vs_source_official": float(metrics["max_dd"] - source_row["official_warm_max_dd"]),
            "daily_path": str(daily_path.relative_to(ROOT)),
            "overlay_config_json": json.dumps(json_safe(overlay_config or {}), sort_keys=True),
        })

fit_metrics = pd.DataFrame(fit_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
official_metrics = pd.DataFrame(metric_rows).sort_values(["variant", "fold"]).reset_index(drop=True)
trace_summary = pd.DataFrame(trace_summary_rows).sort_values(["variant", "fold"]).reset_index(drop=True)
daily_manifest = pd.DataFrame(daily_manifest_rows).sort_values(["variant", "fold"]).reset_index(drop=True)
fit_metrics.to_csv(CARD / "bc_fit_metrics.csv", index=False)
official_metrics.to_csv(CARD / "official_replay_metrics.csv", index=False)
trace_summary.to_csv(CARD / "decision_trace_summary.csv", index=False)
daily_manifest.to_csv(CARD / "official_daily_manifest.csv", index=False)

display(fit_metrics)
display(official_metrics)
display(trace_summary)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[93639:MainThread](2026-05-26 19:36:34,179) WARNING - qlib.data - [data.py:665] - load calendar error: freq=day, future=True; return current calendar!


[93639:MainThread](2026-05-26 19:36:34,182) WARNING - qlib.data - [data.py:668] - You can get future calendar by referring to the following document: https://github.com/microsoft/qlib/blob/main/scripts/data_collector/contrib/README.md


[93639:MainThread](2026-05-26 19:36:34,193) WARNING - qlib.BaseExecutor - [executor.py:121] - `common_infra` is not set for <qlib.backtest.executor.SimulatorExecutor object at 0x13709ea90>


[93639:MainThread](2026-05-26 19:36:34,200) INFO - qlib.backtest caller - [__init__.py:93] - Create new exchange


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,candidate,fold,target_rows,bc_samples,before_accuracy,after_accuracy,after_loss,weight_mean,weight_max
0,multi_obj_floor45_cap20_riskfirst,wf_2022,253,253,0.458498,0.881423,1.025630,1.0,2.490636
1,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,756,755,0.329801,0.765563,1.144428,1.0,2.088680


,candidate,fold,variant,status,requested_start,requested_end,warm_start,warm_end,official_warm_return,official_warm_buyhold_return,...,trough_date,interval_days,interval_avg_pos,interval_max_pos,source_official_relative_return,source_official_max_dd,relative_delta_vs_source_official,max_dd_delta_vs_source_official,daily_path,overlay_config_json
0,multi_obj_floor45_cap20_riskfirst,wf_2022,baseline_no_overlay,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-27.200043,-86.588072,...,2022-11-09,216,0.131131,0.450000,59.388029,30.626799,0.000000,3.552714e-15,runtime/model_selection/card/short_volshock_se...,{}
1,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,baseline_no_overlay,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,29.905886,-0.356635,...,2024-04-19,31,0.222731,0.512402,30.262521,16.122029,0.000000,0.000000e+00,runtime/model_selection/card/short_volshock_se...,{}
2,multi_obj_floor45_cap20_riskfirst,wf_2022,short_volshock_cap05_hit2_no_guard,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-3.023889,-86.588072,...,2022-08-09,151,0.057305,0.450000,59.388029,30.626799,24.176153,-2.187999e+01,runtime/model_selection/card/short_volshock_se...,"{""downside_semivol_21_min"": 0.6404496935293525..."
3,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,short_volshock_cap05_hit2_no_guard,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,29.460629,-0.356635,...,2024-05-01,39,0.137229,0.512402,30.262521,16.122029,-0.445257,-1.014648e+00,runtime/model_selection/card/short_volshock_se...,"{""downside_semivol_21_min"": 0.7493328418395333..."
4,multi_obj_floor45_cap20_riskfirst,wf_2022,short_volshock_cap05_hit2_sell_guard_1pct,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-11.223515,-86.588072,...,2022-08-09,151,0.060954,0.483557,59.388029,30.626799,15.976528,-1.895101e+01,runtime/model_selection/card/short_volshock_se...,"{""de_risk_guard_enabled"": true, ""de_risk_rebal..."
5,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,short_volshock_cap05_hit2_sell_guard_1pct,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,31.963369,-0.356635,...,2024-05-01,39,0.196916,0.512402,30.262521,16.122029,2.057483,-6.041203e-01,runtime/model_selection/card/short_volshock_se...,"{""de_risk_guard_enabled"": true, ""de_risk_rebal..."
6,multi_obj_floor45_cap20_riskfirst,wf_2022,short_volshock_cap05_hit2_sell_guard_zero,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-5.788132,-86.588072,...,2022-08-09,151,0.056833,0.482972,59.388029,30.626799,21.411911,-1.908666e+01,runtime/model_selection/card/short_volshock_se...,"{""de_risk_guard_enabled"": true, ""de_risk_rebal..."
7,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,short_volshock_cap05_hit2_sell_guard_zero,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,31.857110,-0.356635,...,2024-05-01,39,0.196867,0.512402,30.262521,16.122029,1.951224,-6.064981e-01,runtime/model_selection/card/short_volshock_se...,"{""de_risk_guard_enabled"": true, ""de_risk_rebal..."


,candidate,fold,variant,trace_path,trace_rows,overlay_gate_ready_days,overlay_sell_side_de_risk_ready_days,de_risk_rebalance_applied_days,de_risk_trade_lock_applied_days,cap_changed_days,cap_order_days,blocked_by_trade_lock_days,blocked_by_min_rebalance_days,blocked_by_lot_rounding_days,sell_order_days
0,multi_obj_floor45_cap20_riskfirst,wf_2022,baseline_no_overlay,runtime/model_selection/card/short_volshock_se...,308,0,0,0,0,0,0,117,267,0,17
1,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,baseline_no_overlay,runtime/model_selection/card/short_volshock_se...,309,0,0,0,0,0,0,126,260,0,24
2,multi_obj_floor45_cap20_riskfirst,wf_2022,short_volshock_cap05_hit2_no_guard,runtime/model_selection/card/short_volshock_se...,308,236,0,0,0,236,17,50,279,0,13
3,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,short_volshock_cap05_hit2_no_guard,runtime/model_selection/card/short_volshock_se...,309,64,0,0,0,58,8,113,260,0,25
4,multi_obj_floor45_cap20_riskfirst,wf_2022,short_volshock_cap05_hit2_sell_guard_1pct,runtime/model_selection/card/short_volshock_se...,308,236,70,70,18,236,16,28,279,0,14
5,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,short_volshock_cap05_hit2_sell_guard_1pct,runtime/model_selection/card/short_volshock_se...,309,64,12,12,5,58,8,109,257,0,27
6,multi_obj_floor45_cap20_riskfirst,wf_2022,short_volshock_cap05_hit2_sell_guard_zero,runtime/model_selection/card/short_volshock_se...,308,236,32,32,11,236,39,27,256,0,35
7,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,short_volshock_cap05_hit2_sell_guard_zero,runtime/model_selection/card/short_volshock_se...,309,64,8,8,5,58,11,109,254,0,30


In [4]:

summary_rows: list[dict[str, Any]] = []
for variant, group in official_metrics.groupby("variant", sort=True):
    row = {
        "variant": variant,
        "fold_count": int(group["fold"].nunique()),
        "full_gate_count": int(group["full_gate"].sum()),
        "full_gate_all": bool(group["full_gate"].all()),
        "gate_loss": float(group["gate_loss"].sum()),
        "min_relative_return": float(group["official_warm_relative_return"].min()),
        "max_max_dd": float(group["official_warm_max_dd"].max()),
        "mean_avg_pos": float(group["official_warm_avg_pos"].mean()),
        "total_trades": int(group["trades"].sum()),
        "max_dd_delta_vs_source_official": float(group["max_dd_delta_vs_source_official"].max()),
        "mean_relative_delta_vs_source_official": float(group["relative_delta_vs_source_official"].mean()),
    }
    trace_group = trace_summary[trace_summary["variant"].eq(variant)]
    row.update({
        "total_overlay_gate_ready_days": int(trace_group["overlay_gate_ready_days"].sum()) if not trace_group.empty else 0,
        "total_overlay_sell_side_de_risk_ready_days": int(trace_group["overlay_sell_side_de_risk_ready_days"].sum()) if not trace_group.empty else 0,
        "total_de_risk_rebalance_applied_days": int(trace_group["de_risk_rebalance_applied_days"].sum()) if not trace_group.empty else 0,
        "total_de_risk_trade_lock_applied_days": int(trace_group["de_risk_trade_lock_applied_days"].sum()) if not trace_group.empty else 0,
        "total_cap_changed_days": int(trace_group["cap_changed_days"].sum()) if not trace_group.empty else 0,
        "total_cap_order_days": int(trace_group["cap_order_days"].sum()) if not trace_group.empty else 0,
        "total_blocked_by_trade_lock_days": int(trace_group["blocked_by_trade_lock_days"].sum()) if not trace_group.empty else 0,
        "total_blocked_by_min_rebalance_days": int(trace_group["blocked_by_min_rebalance_days"].sum()) if not trace_group.empty else 0,
        "total_sell_order_days": int(trace_group["sell_order_days"].sum()) if not trace_group.empty else 0,
    })
    for fold in WALK_FORWARD_WINDOWS:
        sub = group[group["fold"].eq(fold)]
        if not sub.empty:
            rec = sub.iloc[0]
            row[f"{fold}_relative_return"] = float(rec["official_warm_relative_return"])
            row[f"{fold}_max_dd"] = float(rec["official_warm_max_dd"])
            row[f"{fold}_avg_pos"] = float(rec["official_warm_avg_pos"])
            row[f"{fold}_trades"] = int(rec["trades"])
            row[f"{fold}_full_gate"] = bool(rec["full_gate"])
    summary_rows.append(row)

candidate_summary = pd.DataFrame(summary_rows)
candidate_summary["nonbaseline"] = ~candidate_summary["variant"].eq("baseline_no_overlay")
candidate_summary = candidate_summary.sort_values(
    ["full_gate_all", "nonbaseline", "gate_loss", "total_cap_order_days", "max_max_dd"],
    ascending=[False, False, True, True, True],
).reset_index(drop=True)
candidate_summary.to_csv(CARD / "official_replay_candidate_summary.csv", index=False)
selected = candidate_summary.iloc[0].to_dict()
selected_rows = official_metrics[official_metrics["variant"].eq(str(selected["variant"]))].sort_values("fold")
selected_trace = trace_summary[trace_summary["variant"].eq(str(selected["variant"]))].sort_values("fold")
selected_rows.to_csv(CARD / "selected_official_replay_fold_metrics.csv", index=False)
selected_trace.to_csv(CARD / "selected_decision_trace_summary.csv", index=False)

display(candidate_summary)
display(selected_rows)
display(selected_trace)


,variant,fold_count,full_gate_count,full_gate_all,gate_loss,min_relative_return,max_max_dd,mean_avg_pos,total_trades,max_dd_delta_vs_source_official,...,wf_2022_max_dd,wf_2022_avg_pos,wf_2022_trades,wf_2022_full_gate,wf_2023_2024_relative_return,wf_2023_2024_max_dd,wf_2023_2024_avg_pos,wf_2023_2024_trades,wf_2023_2024_full_gate,nonbaseline
0,short_volshock_cap05_hit2_no_guard,2,1,False,1.256545,29.817263,15.107381,0.135309,78,-1.014648e+00,...,8.746809,0.062850,29,True,29.817263,15.107381,0.207768,49,False,True
1,short_volshock_cap05_hit2_sell_guard_zero,2,1,False,5.155312,32.213744,15.515531,0.138440,107,-6.064981e-01,...,11.540136,0.062714,52,True,32.213744,15.515531,0.214166,55,False,True
2,short_volshock_cap05_hit2_sell_guard_1pct,2,1,False,5.179090,32.320004,15.517909,0.141179,81,-6.041203e-01,...,11.675789,0.067737,29,True,32.320004,15.517909,0.214621,52,False,True
3,baseline_no_overlay,2,0,False,167.488279,30.262521,30.626799,0.193262,90,3.552714e-15,...,30.626799,0.164765,41,False,30.262521,16.122029,0.221759,49,False,False


,candidate,fold,variant,status,requested_start,requested_end,warm_start,warm_end,official_warm_return,official_warm_buyhold_return,...,trough_date,interval_days,interval_avg_pos,interval_max_pos,source_official_relative_return,source_official_max_dd,relative_delta_vs_source_official,max_dd_delta_vs_source_official,daily_path,overlay_config_json
2,multi_obj_floor45_cap20_riskfirst,wf_2022,short_volshock_cap05_hit2_no_guard,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-3.023889,-86.588072,...,2022-08-09,151,0.057305,0.450000,59.388029,30.626799,24.176153,-21.879989,runtime/model_selection/card/short_volshock_se...,"{""downside_semivol_21_min"": 0.6404496935293525..."
3,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,short_volshock_cap05_hit2_no_guard,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,29.460629,-0.356635,...,2024-05-01,39,0.137229,0.512402,30.262521,16.122029,-0.445257,-1.014648,runtime/model_selection/card/short_volshock_se...,"{""downside_semivol_21_min"": 0.7493328418395333..."


,candidate,fold,variant,trace_path,trace_rows,overlay_gate_ready_days,overlay_sell_side_de_risk_ready_days,de_risk_rebalance_applied_days,de_risk_trade_lock_applied_days,cap_changed_days,cap_order_days,blocked_by_trade_lock_days,blocked_by_min_rebalance_days,blocked_by_lot_rounding_days,sell_order_days
2,multi_obj_floor45_cap20_riskfirst,wf_2022,short_volshock_cap05_hit2_no_guard,runtime/model_selection/card/short_volshock_se...,308,236,0,0,0,236,17,50,279,0,13
3,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,short_volshock_cap05_hit2_no_guard,runtime/model_selection/card/short_volshock_se...,309,64,0,0,0,58,8,113,260,0,25


In [5]:

selected_variant = str(selected["variant"])
selected_full_gate = bool(selected["full_gate_all"])
selected_min_relative = float(selected["min_relative_return"])
selected_max_dd = float(selected["max_max_dd"])
selected_gate_loss = float(selected["gate_loss"])
selected_total_cap_orders = int(selected.get("total_cap_order_days", 0))
selected_total_gate_days = int(selected.get("total_overlay_gate_ready_days", 0))
selected_total_de_risk_days = int(selected.get("total_overlay_sell_side_de_risk_ready_days", 0))
baseline = candidate_summary[candidate_summary["variant"].eq("baseline_no_overlay")].iloc[0].to_dict()

selected_has_guard = "sell_guard" in selected_variant
if selected_full_gate and selected_variant != "baseline_no_overlay":
    decision = "official_short_volshock_sell_guard_full_gate_needs_fixed_split_no_promotion"
    next_action = "run fixed-split plus wf_2020 actor-retention screen with the same sell-side guard before any promotion"
    conclusion = [
        f"Official warm-start replay confirms the sell-side guard candidate: {selected_variant} clears both folds with min relative={selected_min_relative:.2f}pp and max MaxDD={selected_max_dd:.2f}%.",
        f"Action semantics were active: overlay gate days={selected_total_gate_days}, de-risk ready days={selected_total_de_risk_days}, cap order days={selected_total_cap_orders}; qlib costs and lot rounding are included by backtest_profile defaults.",
        "No promotion and no actor retention from this card because the BC actors are regenerated diagnostics and fixed-split/wf_2020 evidence is still missing.",
        f"Next action: {next_action}.",
    ]
elif selected_has_guard:
    decision = "official_short_volshock_sell_guard_failed_no_promotion"
    next_action = "use the trace to decide whether failure is still execution absorption or whether the signal belongs in checkpoint scoring / target generation"
    conclusion = [
        f"The best candidate uses the sell-side guard but still misses the joint gate: selected={selected_variant}, min relative={selected_min_relative:.2f}pp, max MaxDD={selected_max_dd:.2f}%, gate_loss={selected_gate_loss:.2f}.",
        f"The guard produced de-risk ready days={selected_total_de_risk_days} and cap order days={selected_total_cap_orders}; this separates execution absorption from model-selection/target-generation quality.",
        f"Next action: {next_action}.",
    ]
else:
    decision = "official_short_volshock_sell_guard_no_improvement_no_promotion"
    next_action = "move short_volshock into checkpoint scoring or target generation; do not continue threshold-only execution guard tuning"
    conclusion = [
        f"No sell-side guard variant beats the no-guard official replay on the joint gate; selected={selected_variant}, min relative={selected_min_relative:.2f}pp, max MaxDD={selected_max_dd:.2f}%, gate_loss={selected_gate_loss:.2f}.",
        "Execution absorption was not the only bottleneck; the remaining issue belongs in checkpoint scoring / target generation rather than another cap threshold screen.",
        f"Next action: {next_action}.",
    ]

selection_payload = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "promotion_allowed": False,
    "actor_retention": "none; regenerated diagnostic BC actors only",
    "selected_variant": json_safe(selected),
    "selected_fold_metrics": selected_rows.to_dict("records"),
    "selected_decision_trace_summary": selected_trace.to_dict("records"),
    "baseline_variant": json_safe(baseline),
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
    "next_action": next_action,
}
summary_payload = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "canonical_baseline": source_summary["canonical_baseline"],
    "direct_comparison_baseline": [
        "short_horizon_risk_cap_counterfactual_v1",
        "short_volshock_official_replay_screen_v1",
    ],
    "profile": PROFILE_NAME,
    "role": "diagnose",
    "changed_layer": ["execution", "model_selection", "risk_state_representation"],
    "external_sources": EXTERNAL_SOURCES,
    "closest_historical_success_case": "short_volshock_official_replay_screen_v1 showed cap=0.05 min_hits=2 passed wf_2022 and near-missed wf_2023_2024 under official replay",
    "closest_historical_failure_case": "short_volshock_official_replay_screen_v1 showed cap_changed_days greatly exceeded cap_order_days because min_rebalance and trade_lock absorbed sell-side de-risk actions",
    "duplicate_check": "This run keeps the same official warm-start replay but changes only the execution semantics by adding a default-off sell-side de-risk guard to the short_volshock action cap.",
    "selected_variant": json_safe(selected),
    "selected_fold_metrics": selected_rows.to_dict("records"),
    "selected_decision_trace_summary": selected_trace.to_dict("records"),
    "baseline_variant": json_safe(baseline),
    "promotion_allowed": False,
    "ppo_screen_allowed_next": False,
    "decision": decision,
    "conclusion": conclusion,
    "sidecars": [
        "warm_start_segments.csv",
        "bc_fit_metrics.csv",
        "official_replay_metrics.csv",
        "decision_trace_summary.csv",
        "official_replay_candidate_summary.csv",
        "selected_official_replay_fold_metrics.csv",
        "selected_decision_trace_summary.csv",
        "official_daily_manifest.csv",
        "daily_paths/*.csv",
        "decision_traces/*.csv",
        "selection.json",
        "summary.json",
    ],
}
write_json(CARD / "selection.json", selection_payload)
write_json(CARD / "summary.json", summary_payload)

conclusion_md = "\n".join(f"- {item}" for item in conclusion)
display(Markdown(f"""
## Conclusion

{conclusion_md}

### Selected Official Replay Variant

- variant: `{selected_variant}`
- min relative: `{selected_min_relative:.2f}pp`
- max MaxDD: `{selected_max_dd:.2f}%`
- total trades: `{int(selected.get('total_trades', 0))}`
- overlay gate days: `{selected_total_gate_days}`
- de-risk ready days: `{selected_total_de_risk_days}`
- cap order days: `{selected_total_cap_orders}`
- promotion: `False`
"""))
display(selected_rows[["fold", "variant", "official_warm_relative_return", "official_warm_max_dd", "official_warm_avg_pos", "trades", "full_gate", "gate_loss", "interval_avg_pos"]])
display(selected_trace)



## Conclusion

- No sell-side guard variant beats the no-guard official replay on the joint gate; selected=short_volshock_cap05_hit2_no_guard, min relative=29.82pp, max MaxDD=15.11%, gate_loss=1.26.
- Execution absorption was not the only bottleneck; the remaining issue belongs in checkpoint scoring / target generation rather than another cap threshold screen.
- Next action: move short_volshock into checkpoint scoring or target generation; do not continue threshold-only execution guard tuning.

### Selected Official Replay Variant

- variant: `short_volshock_cap05_hit2_no_guard`
- min relative: `29.82pp`
- max MaxDD: `15.11%`
- total trades: `78`
- overlay gate days: `300`
- de-risk ready days: `0`
- cap order days: `25`
- promotion: `False`


,fold,variant,official_warm_relative_return,official_warm_max_dd,official_warm_avg_pos,trades,full_gate,gate_loss,interval_avg_pos
2,wf_2022,short_volshock_cap05_hit2_no_guard,83.564183,8.746809,0.062850,29,True,0.000000,0.057305
3,wf_2023_2024,short_volshock_cap05_hit2_no_guard,29.817263,15.107381,0.207768,49,False,1.256545,0.137229


,candidate,fold,variant,trace_path,trace_rows,overlay_gate_ready_days,overlay_sell_side_de_risk_ready_days,de_risk_rebalance_applied_days,de_risk_trade_lock_applied_days,cap_changed_days,cap_order_days,blocked_by_trade_lock_days,blocked_by_min_rebalance_days,blocked_by_lot_rounding_days,sell_order_days
2,multi_obj_floor45_cap20_riskfirst,wf_2022,short_volshock_cap05_hit2_no_guard,runtime/model_selection/card/short_volshock_se...,308,236,0,0,0,236,17,50,279,0,13
3,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,short_volshock_cap05_hit2_no_guard,runtime/model_selection/card/short_volshock_se...,309,64,0,0,0,58,8,113,260,0,25
